In [16]:
import sys
import pickle
sys.path.append('../../TaskExecutionTimeMining/')
from divide_and_conquer import *
from sklearn.feature_selection import mutual_info_regression

import numpy as np
np.seterr(divide='ignore', invalid='ignore')

{'divide': 'ignore', 'over': 'warn', 'under': 'ignore', 'invalid': 'ignore'}

In [17]:
with open("../transformed_event_logs/artificial_start_end_2.pickle", "rb") as f:
    event_log = pickle.load(f)

# numerical attributes : duration, seconds_in_day, day_in_week
numerical_attributes = [
    'duration_seconds',
    'seconds_in_day',
    #'day_of_week',
]

transformed_event_log = event_log.copy()

for num_attr in numerical_attributes:
    transformed_event_log[num_attr] = np.log(transformed_event_log[num_attr]+1)
    transformed_event_log[num_attr] = (transformed_event_log[num_attr] - transformed_event_log[num_attr].mean()) / transformed_event_log[num_attr].std()

In [18]:
transformed_event_log

,concept:name_start,time:timestamp_start,org:resource_start,case:concept:name,id_start,concept:name_complete,time:timestamp_complete,org:resource_complete,id_complete,duration,...,duration_ms,duration_hours,seconds_in_day,DIAGNOSIS,REPAIR,1,Clark,Jane,Joe,Karsten
0,DIAGNOSIS,2020-01-02 19:17:34.416296+00:00,Karsten,0,0,REPAIR,2020-01-02 20:14:49.089245+00:00,1,1,0 days 00:57:14.672949,...,3.434673e+06,0.954076,0.762380,1,0,0,0,0,0,1
1,REPAIR,2020-01-02 20:14:49.089245+00:00,1,0,1,QUALITY_CONTROL,2020-01-03 05:52:47.939823+00:00,Karsten,2,0 days 09:37:58.850578,...,3.467885e+07,9.633014,0.809689,1,1,1,0,0,0,1
2,DIAGNOSIS,2020-01-02 21:52:35.878255+00:00,Karsten,1,3,REPAIR,2020-01-02 22:58:13.130533+00:00,Clark,4,0 days 01:05:37.252278,...,3.937252e+06,1.093681,0.885548,1,0,0,0,0,0,1
3,REPAIR,2020-01-02 22:58:13.130533+00:00,Clark,1,4,QUALITY_CONTROL,2020-01-03 08:46:41.828621+00:00,Joe,5,0 days 09:48:28.698088,...,3.530870e+07,9.807972,0.933368,1,1,0,1,0,0,1
4,DIAGNOSIS,2020-01-03 04:00:20.210312+00:00,Karsten,2,6,REPAIR,2020-01-03 04:59:01.560453+00:00,Jane,7,0 days 00:58:41.350141,...,3.521350e+06,0.978153,-0.778256,1,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1773,REPAIR,2022-06-29 12:39:51.404397+00:00,Clark,886,2659,QUALITY_CONTROL,2022-06-29 22:49:12.130581+00:00,1,2660,0 days 10:09:20.726184,...,3.656073e+07,10.155757,0.349830,1,1,0,1,1,0,0
1774,DIAGNOSIS,2022-06-29 20:30:50.890143+00:00,Joe,887,2661,REPAIR,2022-06-29 21:46:11.242668+00:00,Joe,2662,0 days 01:15:20.352525,...,4.520353e+06,1.255653,0.822526,1,0,0,0,0,1,0
1775,REPAIR,2022-06-29 21:46:11.242668+00:00,Joe,887,2662,QUALITY_CONTROL,2022-06-30 03:14:21.565405+00:00,Karsten,2663,0 days 05:28:10.322737,...,1.969032e+07,5.469534,0.880758,1,1,0,0,0,2,0
1776,DIAGNOSIS,2022-07-01 01:26:36.080967+00:00,Clark,888,2664,REPAIR,2022-07-01 02:04:45.218753+00:00,Joe,2665,0 days 00:38:09.137786,...,2.289138e+06,0.635872,-1.778501,1,0,0,1,0,0,0


In [19]:
target_column = 'duration_seconds'
continuous_feature_columns = ['seconds_in_day']
nominal_feature_columns = ['concept:name_start', 'org:resource_start']

In [20]:
mi_matrix = calculate_mi_matrix(transformed_event_log, target_column, continuous_feature_columns, nominal_feature_columns,
                                verbose=True)

Computing MI:   0%|          | 0/9 [00:00<?, ?it/s]

MI(concept:name_start, concept:name_start) = 0.9999999999971146
MI(org:resource_start, org:resource_start) = 2.3212780411953875
MI(concept:name_start, org:resource_start) = 0.0019516643055862276
MI(seconds_in_day, seconds_in_day) = 1.6129658482308231
MI(org:resource_start, seconds_in_day) = 0.06309701455516845
MI(org:resource_start, duration_seconds) = 0.0630399952582738
MI(concept:name_start, seconds_in_day) = 0.05099903573520534
MI(concept:name_start, duration_seconds) = 0.6380978458191311
MI(seconds_in_day, duration_seconds) = 0.05511945658038969


In [21]:
mimr, all_relevance = calculate_maximal_relevance_minimal_redundancy_split(mi_matrix, target_column, continuous_feature_columns, nominal_feature_columns)
print(mimr)
print(all_relevance)

concept:name_start
{'concept:name_start': np.float64(0.28711427913982907), 'org:resource_start': np.float64(-0.7324022447604404), 'seconds_in_day': np.float64(-0.520567842926676)}
